# Capítulo 4 · Fundamentos de MCP

Las herramientas de la Clínica Veterinaria Naranco, antes dentro del programa del agente y después publicadas en tres servidores MCP: el de la clínica, el de la facturación y el de la farmacia, que es un programa aparte con el que se habla por stdio. Se ven los mensajes JSON-RPC de una conversación entre cliente y servidor, se mide el agente con dieciséis tareas en cuatro variantes y se mide lo que cuesta el formato del protocolo en tokens.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo y la clínica

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 4: fundamentos de MCP. Las herramientas de la clínica, antes dentro del
programa y después publicadas en tres servidores MCP que el agente descubre."""
import asyncio
import copy
import json
import re
import time
import subprocess
import sys
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def reservar_cita(mascota, veterinario, fecha, hora, motivo):
    """Reserva un hueco libre de la agenda; devuelve la cita o el problema."""
    if mascota not in MASCOTAS:
        return {"error": f"No existe la mascota {mascota}."}
    if veterinario not in VETERINARIOS:
        return {"error": f"No existe el veterinario {veterinario}."}
    if any(c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == veterinario
           and c["estado"] != "cancelada" for c in CITAS):
        return {"error": f"{VETERINARIOS[veterinario]} ya tiene una cita el {fecha} "
                         f"a las {hora}."}
    cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
            "veterinario": veterinario, "mascota": mascota, "motivo": motivo,
            "estado": "pendiente"}
    CITAS.append(cita)
    return cita


def cancelar_cita(cita):
    """Cancela una cita pendiente."""
    for c in CITAS:
        if c["id"] == cita:
            if c["estado"] != "pendiente":
                return {"error": f"La cita {cita} está «{c['estado']}»."}
            c["estado"] = "cancelada"
            return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
    return {"error": f"No existe la cita {cita}."}


def avisar(dueno, texto):
    """Envía un aviso al dueño (aquí se guarda en una lista)."""
    if dueno not in DUENOS:
        return {"error": f"No existe el cliente {dueno}."}
    AVISOS.append({"dueno": dueno, "texto": texto})
    return {"enviado": True, "a": DUENOS[dueno]["nombre"],
            "por": DUENOS[dueno]["avisos"]}


def edad(nacimiento):
    """Años cumplidos el día de los datos."""
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    """Lo esencial de una mascota, sin el historial completo."""
    m = MASCOTAS[mid]
    return {"id": mid, "nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida",
            "dueno": {"id": m["dueno"], "nombre": DUENOS[m["dueno"]]["nombre"]}}


def vacunas_con_estado(mid):
    """Cada vacuna con su próxima fecha y si ya está vencida."""
    return [{"vacuna": v["vacuna"], "proxima": v["proxima"],
             "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"}
            for v in VACUNAS[mid]]


def ficha_cliente_3(id_cliente):
    if id_cliente not in DUENOS:
        return {"error": f"No existe el cliente {id_cliente}. Búscalo por nombre con "
                         "buscar(tabla='duenos', campo='nombre', valor=...)."}
    pendientes = [f for f in FACTURAS if f["dueno"] == id_cliente and not f["pagada"]]
    return {"id": id_cliente, "nombre": DUENOS[id_cliente]["nombre"],
            "avisos_por": DUENOS[id_cliente]["avisos"],
            "mascotas": [{"id": m, "nombre": d["nombre"], "especie": d["especie"]}
                         for m, d in MASCOTAS.items() if d["dueno"] == id_cliente],
            "facturas_sin_pagar": [{"id": f["id"], "fecha": f["fecha"], "total": f["total"]}
                                   for f in pendientes],
            "total_sin_pagar": round(sum(f["total"] for f in pendientes), 2)}


def agenda_3(fecha):
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(fecha)):
        return {"error": f"La fecha «{fecha}» no es válida; usa AAAA-MM-DD."}
    citas = [c for c in CITAS if c["fecha"] == fecha and c["estado"] != "cancelada"]
    return {"fecha": fecha, "citas": [f"{c['id']} {c['hora']} {c['veterinario']} "
                                      f"{MASCOTAS[c['mascota']]['nombre']} ({c['mascota']})"
                                      for c in citas],
            "veterinarios": VETERINARIOS,
            "horas_libres": {v: [h for h in HORAS if not any(
                c["hora"] == h and c["veterinario"] == v for c in citas)]
                for v in VETERINARIOS}}


def localizar(mascota, dueno=""):
    """El id de la mascota a partir de su nombre (y el del dueño, si hay varias)."""
    ids = [m for m, d in MASCOTAS.items() if sin_tildes(mascota) == sin_tildes(d["nombre"])
           and sin_tildes(dueno) in sin_tildes(DUENOS[d["dueno"]]["nombre"])]
    if not ids:
        return None, {"error": f"No hay ninguna mascota llamada «{mascota}»"
                               + (f" de {dueno}" if dueno else "") + " en la clínica."}
    if len(ids) > 1:
        return None, {"error": f"Hay {len(ids)} mascotas llamadas «{mascota}»; indica el "
                               "nombre del dueño.",
                      "mascotas": [resumen_mascota(m) for m in ids]}
    return ids[0], None


def veterinario_de(nombre):
    """El id del veterinario a partir de su nombre o de su id."""
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or \
                sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def buscar_mascota(nombre, dueno=""):
    mid, error = localizar(nombre, dueno)
    return error or resumen_mascota(mid)


def mascotas_de(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{"nombre": DUENOS[d]["nombre"], "mascotas": [
        {"nombre": x["nombre"], "especie": x["especie"], "raza": x["raza"]}
        for m, x in MASCOTAS.items() if x["dueno"] == d]} for d in ids]}


def vacunas(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(),
                     "vacunas": vacunas_con_estado(mid)}


def historial(mascota, ultimas=3, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    return {"mascota": MASCOTAS[mid]["nombre"], "visitas": [
        {k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
        for v in HISTORIAL[mid][-int(ultimas):]]}


def dosis(medicamento, mascota, dueno=""):
    med = FARMACIA.get(sin_tildes(medicamento))
    if med is None:
        return {"error": f"No tenemos «{medicamento}». Medicamentos: "
                         + ", ".join(FARMACIA) + "."}
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    m = MASCOTAS[mid]
    avisos = []
    if sin_tildes(medicamento) in [sin_tildes(a) for a in m["alergias"]]:
        avisos.append(f"{m['nombre']} es alérgico a {medicamento}: no recetar.")
    if m["especie"] not in med["especies"]:
        avisos.append(f"{medicamento} no está indicado en {m['especie']}s.")
    return {"medicamento": sin_tildes(medicamento), "mascota": m["nombre"],
            "peso_kg": m["peso_kg"], "dosis_mg_por_kg": med["dosis_mg_kg"],
            "dosis_mg_por_toma": round(med["dosis_mg_kg"] * m["peso_kg"], 2),
            "existencias": med["existencias"],
            "reposicion": med.get("reposicion", "sin fecha") if not med["existencias"] else None,
            "avisos": avisos or "ninguno"}


def huecos_libres(fecha, veterinario=""):
    r = agenda_3(fecha)
    if "error" in r:
        return r
    vets = [veterinario_de(veterinario)] if veterinario else list(VETERINARIOS)
    if vets == [None]:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    return {"fecha": fecha, "horas_libres": {VETERINARIOS[v]: r["horas_libres"][v]
                                             for v in vets}}


def citas_de(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
        {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
         "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
        for c in CITAS if c["mascota"] == mid]}


def cuentas(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{k: v for k, v in ficha_cliente_3(d).items() if k != "id"}
                         for d in ids]}


def reservar(mascota, veterinario, fecha, hora, motivo, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    vet = veterinario_de(veterinario)
    if vet is None:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    r = reservar_cita(mid, vet, fecha, hora, motivo)
    if "error" in r:
        return r
    return {"reservada": r["id"], "mascota": MASCOTAS[mid]["nombre"],
            "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}


def avisar_dueno_de(mascota, texto, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or avisar(MASCOTAS[mid]["dueno"], texto)


JUEGO_TAREAS = {"buscar_mascota": buscar_mascota, "mascotas_de": mascotas_de,
                "vacunas": vacunas, "historial": historial, "dosis": dosis,
                "huecos_libres": huecos_libres, "citas_de": citas_de, "cuentas": cuentas,
                "reservar_cita": reservar, "cancelar_cita": cancelar_cita,
                "avisar_dueno": avisar_dueno_de}

In [ ]:
def herramienta(nombre, descripcion, parametros):
    """Esquema de una herramienta; los parámetros sin valor por defecto son obligatorios."""
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object",
                       "required": [p for p, (_, oblig) in parametros.items() if oblig],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parametros.items()}}}}


MASCOTA = ("Nombre de la mascota.", True)


DUENO = ("Nombre del dueño; solo si hay varias mascotas con ese nombre.", False)


ESQUEMAS_TAREAS = [
    herramienta("buscar_mascota", "Datos de una mascota buscada por su nombre: especie, "
                "edad, peso actual, alergias y dueño.", {"nombre": MASCOTA, "dueno": DUENO}),
    herramienta("mascotas_de", "Las mascotas de un cliente, buscado por su nombre.",
                {"dueno": ("Nombre del dueño.", True)}),
    herramienta("vacunas", "Vacunas de una mascota: la próxima fecha de cada una y si "
                "está vencida.", {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("historial", "Las últimas visitas de una mascota: fecha, motivo, "
                "tratamiento y peso.", {"mascota": MASCOTA,
                                        "ultimas": ("Cuántas visitas (por defecto, 3).", False),
                                        "dueno": DUENO}),
    herramienta("dosis", "Dosis de un medicamento para una mascota según su peso actual, "
                "con avisos de alergia o especie y las existencias.",
                {"medicamento": ("Nombre del medicamento.", True), "mascota": MASCOTA,
                 "dueno": DUENO}),
    herramienta("huecos_libres", "Horas libres de la agenda en un día, de un veterinario o "
                "de todos.", {"fecha": ("AAAA-MM-DD.", True),
                              "veterinario": ("Nombre del veterinario; vacío para todos.", False)}),
    herramienta("citas_de", "Citas de una mascota (pasadas y pendientes), con su id.",
                {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("cuentas", "Facturas sin pagar de un cliente, buscado por su nombre, y el "
                "total que debe.", {"dueno": ("Nombre del dueño.", True)}),
    herramienta("reservar_cita", "Reserva una cita en una hora libre.",
                {"mascota": MASCOTA, "veterinario": ("Nombre del veterinario.", True),
                 "fecha": ("AAAA-MM-DD.", True), "hora": ("HH:MM, en punto o y media.", True),
                 "motivo": ("Motivo de la cita.", True), "dueno": DUENO}),
    herramienta("cancelar_cita", "Cancela una cita pendiente.",
                {"cita": ("Id de la cita, como CT-0026 (lo da citas_de).", True)}),
    herramienta("avisar_dueno", "Envía un aviso al dueño de una mascota, por correo o SMS "
                "según prefiera.", {"mascota": MASCOTA, "texto": ("Texto del aviso, en "
                                                                   "español.", True),
                                     "dueno": DUENO}),
]


SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")


def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def ejecutar(llamada, funciones):
    """Ejecuta una llamada si la herramienta existe y los argumentos encajan."""
    funcion = funciones.get(llamada["name"])
    if funcion is None:
        return {"error": f"No existe la herramienta {llamada['name']}."}
    try:
        return funcion(**llamada["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Argumentos no válidos: {e}"}


def agente(tarea, funciones, esquemas, max_pasos=8, ver=False):
    """Pensar, actuar, observar y repetir. Cuenta los tokens de los resultados."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, esquemas)
        llamadas = leer_llamadas(respuesta)[:1]            # una cada vez
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "tokens_salida": salida,
                      "pensamiento": pensamiento, "llamadas": []})
        if not llamadas:
            traza[-1]["respuesta"] = respuesta
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        if ver and pensamiento:
            print(f"[{paso}] {pensamiento}")
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = ejecutar(llamada, funciones)
            texto = json.dumps(resultado, ensure_ascii=False)
            traza[-1]["llamadas"].append({**llamada, "resultado": resultado,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza


def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def fecha_en(respuesta, iso):
    """¿Aparece la fecha, escrita como 2027-08-22, 22/08/2027 o 22 de agosto de 2027?"""
    a, m, d = iso.split("-")
    meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
             "septiembre", "octubre", "noviembre", "diciembre"]
    return dice(respuesta, iso, rf"\b0?{int(d)}/0?{int(m)}/{a}",
                rf"\b0?{int(d)} de {meses[int(m) - 1]}( de {a})?")


def citas_nuevas():
    return [c for c in CITAS if c["id"] not in {x["id"] for x in CITAS0}]


def primera_libre(fecha, vet):
    ocupadas = {c["hora"] for c in CITAS0 if c["fecha"] == fecha and c["veterinario"] == vet}
    return next(h for h in HORAS if h not in ocupadas)


def debe(nombre):
    return round(sum(f["total"] for f in FACTURAS if not f["pagada"]
                     and DUENOS[f["dueno"]]["nombre"] == nombre), 2)


def importe(respuesta, valor):
    """¿Aparece el importe, como 136, 136,00 o 136.00?"""
    entero = f"{valor:.2f}".replace(".00", "")
    return dice(respuesta, rf"\b{entero}([,.]00?)?\b", f"{valor:.2f}".replace(".", ","))


def aviso_a(nombre, *formas):
    return any(DUENOS[a["dueno"]]["nombre"] == nombre
               and all(re.search(f, a["texto"], re.I) for f in formas) for a in AVISOS)


TAREAS = [
    ("vacuna_rabia", "¿Cuándo le toca a Lola, la perra de Marta Fernández, la próxima "
     "vacuna de la rabia?",
     lambda r: fecha_en(r, "2027-08-22")),
    ("dosis", "¿Cuántos miligramos de meloxicam le corresponden a Max en cada toma?",
     lambda r: dice(r, r"2[,.]07", r"2[,.]1\b")),
    ("alergia", "Van a recetarle amoxicilina a Simba. ¿Hay algún problema?",
     lambda r: dice(r, r"al[eé]rg")),
    ("reservar", "Reserva para Toby una revisión con el Dr. Pablo Menéndez el jueves 22, "
     "en la primera hora libre.",
     lambda r: [(c["mascota"], c["veterinario"], c["fecha"], c["hora"]) for c in citas_nuevas()]
     == [("M-002", "V2", "2026-10-22", primera_libre("2026-10-22", "V2"))]),
    ("cancelar", "Cancela la cita que tiene Rocky mañana.",
     lambda r: [c["id"] for c in CITAS if c["estado"] == "cancelada"] == ["CT-0060"]),
    ("vencidas_aviso", "Avisa al dueño de Toby de las vacunas que tiene vencidas.",
     lambda r: aviso_a("Pedro Álvarez", "rabia", "polivalente", "leishmaniosis")),
    ("deuda", "¿Cuánto debe Óscar Prieto en facturas sin pagar?",
     lambda r: importe(r, debe("Óscar Prieto"))),
    ("gata_de_marta", "Marta Fernández llama por su gata. ¿Tiene alguna vacuna vencida?",
     lambda r: dice(r, "trivalente") and dice(r, "vencid|caducad|atrasad|pasad")),
    ("peso", "¿Cuánto pesaba Zeus en su última visita, y cuánto ha cambiado respecto a "
     "la anterior?",
     lambda r: dice(r, r"32[,.]2") and dice(r, r"0[,.]6")
     and not dice(r, r"últim[ao] visita[^.,]{0,30}(2025|31[,.]6)", r"31[,.]6 kg en (su|la) últim")),
    ("ultima_visita", "¿Por qué vino Nube la última vez y qué tratamiento le pusieron?",
     lambda r: dice(r, "herida") and dice(r, "amoxicilina")),
    ("huecos", "¿Qué horas tiene libres la Dra. Ana Suárez el miércoles 21 por la tarde?",
     lambda r: all(dice(r, h) for h in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not dice(r, r"17:00")),
    ("no_existe", "¿Cuándo le toca la vacuna de la rabia a Pancho?",
     lambda r: dice(r, r"no (hay|existe|encuentr|figura|tenemos|aparece)", "ning[uú]n")
     and not dice(r, r"\b20\d\d-\d\d-\d\d\b")),
]

## 2. Antes: todo dentro del programa

Una copia de las dos funciones de la farmacia, con sus esquemas escritos a mano.

In [ ]:
def raices(texto):
    """Las siete primeras letras de cada palabra, sin tildes, para comparar nombres."""
    texto = texto.lower().translate(str.maketrans("áéíóúü", "aeiouu"))
    return {p[:7] for p in re.findall(r"\w{4,}", texto)}


def precio(servicio):
    """Precio en euros de un servicio de la clínica o de un medicamento."""
    precios = {**TARIFAS, **{k: d["precio"] for k, d in FARMACIA.items()}}
    comunes = {k: len(raices(servicio) & raices(k)) for k in precios}
    mejor = max(comunes.values())
    if mejor == 0:
        return {"error": f"No hay ninguna tarifa de «{servicio}». Servicios: "
                         + ", ".join(TARIFAS) + "; y los medicamentos de la farmacia."}
    return {"precios_euros": {k: precios[k] for k, c in comunes.items() if c == mejor}}


def existencias(medicamento):
    """Unidades de un medicamento en la farmacia y, si no quedan, cuándo llegan."""
    clave = next((k for k in FARMACIA if raices(k) & raices(medicamento)), None)
    if clave is None:
        return {"error": f"No tenemos «{medicamento}». Medicamentos: " + ", ".join(FARMACIA) + "."}
    m = FARMACIA[clave]
    return {"medicamento": clave, "presentacion": m["presentacion"], "existencias": m["existencias"],
            "reposicion": m.get("reposicion", "sin fecha") if not m["existencias"] else None}


FUNCIONES = {**JUEGO_TAREAS, "precio": precio, "existencias": existencias}


ESQUEMAS = ESQUEMAS_TAREAS + [
    herramienta("precio", "Precio en euros de un servicio (consulta, vacunas, análisis…) o de "
                "un medicamento.", {"servicio": ("Servicio o medicamento.", True)}),
    herramienta("existencias", "Unidades de un medicamento en la farmacia y, si no quedan, "
                "cuándo llegan.", {"medicamento": ("Nombre del medicamento.", True)})]


def cargar_datos():
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS
    global VETERINARIOS, HORAS, TARIFAS
    MASCOTAS, DUENOS = descargar("mascotas.json"), descargar("duenos.json")
    HISTORIAL, VACUNAS = descargar("historial.json"), descargar("vacunas.json")
    CITAS0, FARMACIA = descargar("citas.json"), descargar("farmacia.json")
    FACTURAS, VETERINARIOS = descargar("facturas.json"), descargar("veterinarios.json")
    TARIFAS = descargar("tarifas.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reiniciar()

In [ ]:
cargar_datos()
print(existencias("doxiciclina"))

## 3. El servidor de la farmacia, un programa aparte

Se guarda en un archivo para poder lanzarlo como un proceso.

In [ ]:
%%writefile servidor_farmacia.py
"""Servidor MCP de la farmacia de la clínica: precios y existencias.

Es un programa aparte, como si lo hubiera escrito otro equipo: no comparte código con el
agente. Se comunica por la entrada y la salida estándar (stdio), con mensajes JSON-RPC.
Uso: python servidor_farmacia.py [carpeta_de_datos]
"""
import json
import re
import sys
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from pydantic import Field

DATOS = Path(sys.argv[1] if len(sys.argv) > 1 else "datos")
FARMACIA = json.loads((DATOS / "farmacia.json").read_text(encoding="utf-8"))
TARIFAS = json.loads((DATOS / "tarifas.json").read_text(encoding="utf-8"))

farmacia = MCPServer("farmacia", instructions="Precios y existencias de la clínica.")


def raices(texto):
    """Las siete primeras letras de cada palabra, sin tildes, para comparar nombres."""
    texto = texto.lower().translate(str.maketrans("áéíóúü", "aeiouu"))
    return {p[:7] for p in re.findall(r"\w{4,}", texto)}


@farmacia.tool()
def precio(servicio: Annotated[str, Field(description="Servicio o medicamento.")]) -> dict:
    """Precio en euros de un servicio (consulta, vacunas, análisis…) o de un medicamento."""
    precios = {**TARIFAS, **{k: d["precio"] for k, d in FARMACIA.items()}}
    comunes = {k: len(raices(servicio) & raices(k)) for k in precios}
    mejor = max(comunes.values())
    if mejor == 0:
        return {"error": f"No hay ninguna tarifa de «{servicio}». Servicios: "
                         + ", ".join(TARIFAS) + "; y los medicamentos de la farmacia."}
    return {"precios_euros": {k: precios[k] for k, c in comunes.items() if c == mejor}}


@farmacia.tool()
def existencias(medicamento: Annotated[str, Field(description="Nombre del medicamento.")]) -> dict:
    """Unidades de un medicamento en la farmacia y, si no quedan, cuándo llegan."""
    clave = next((k for k in FARMACIA if raices(k) & raices(medicamento)), None)
    if clave is None:
        return {"error": f"No tenemos «{medicamento}». Medicamentos: " + ", ".join(FARMACIA) + "."}
    m = FARMACIA[clave]
    return {"medicamento": clave, "presentacion": m["presentacion"], "existencias": m["existencias"],
            "reposicion": m.get("reposicion", "sin fecha") if not m["existencias"] else None}


if __name__ == "__main__":
    farmacia.run()          # por defecto, stdio

## 4. Lo que viaja por el cable

Los mensajes JSON-RPC, escritos a mano, una línea cada uno.

In [ ]:
# La farmacia es otro programa, servidor_farmacia.py; se lanza como un proceso aparte y
# se habla con él por su entrada y su salida estándar (stdio).
SERVIDOR_FARMACIA = Path("servidor_farmacia.py")      # en el cuaderno, en la carpeta de trabajo


farmacia = StdioServerParameters(command=sys.executable,
                                 args=[str(SERVIDOR_FARMACIA), str(DATOS)])

In [ ]:
def conversacion_cruda():
    """Habla con el servidor de la farmacia escribiendo a mano los mensajes JSON-RPC, una
    línea por mensaje, y devuelve lo que se envía (>>) y lo que se recibe (<<)."""
    proceso = subprocess.Popen(
        [sys.executable, str(SERVIDOR_FARMACIA), str(DATOS)], stdin=subprocess.PIPE,
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, encoding="utf-8")
    registro = []

    def enviar(mensaje, respuesta=True):
        registro.append((">>", mensaje))
        proceso.stdin.write(json.dumps(mensaje, ensure_ascii=False) + "\n")
        proceso.stdin.flush()
        if respuesta:
            registro.append(("<<", json.loads(proceso.stdout.readline())))

    enviar({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
        "protocolVersion": "2025-06-18", "capabilities": {},
        "clientInfo": {"name": "recepcion", "version": "1.0"}}})
    enviar({"jsonrpc": "2.0", "method": "notifications/initialized"}, respuesta=False)
    enviar({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
    enviar({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {
        "name": "existencias", "arguments": {"medicamento": "doxiciclina"}}})
    proceso.stdin.close()
    proceso.wait(timeout=30)
    return registro

In [ ]:
for direccion, mensaje in conversacion_cruda():
    print(direccion, json.dumps(mensaje, ensure_ascii=False)[:400])

## 5. Los servidores de la clínica y de la facturación

In [ ]:
Mascota = Annotated[str, Field(description="Nombre de la mascota.")]


Dueno = Annotated[str, Field(description="Nombre del dueño; solo si hay varias mascotas "
                                         "con ese nombre.")]


Fecha = Annotated[str, Field(description="AAAA-MM-DD.")]


clinica = MCPServer("clinica", instructions="Mascotas, vacunas, historial, agenda y avisos "
                    "de la Clínica Veterinaria Naranco.")


@clinica.tool(name="buscar_mascota")
def mcp_buscar_mascota(nombre: Mascota, dueno: Dueno = "") -> dict:
    """Datos de una mascota buscada por su nombre: especie, edad, peso actual, alergias y dueño."""
    return buscar_mascota(nombre, dueno)


@clinica.tool(name="mascotas_de")
def mcp_mascotas_de(dueno: Annotated[str, Field(description="Nombre del dueño.")]) -> dict:
    """Las mascotas de un cliente, buscado por su nombre."""
    return mascotas_de(dueno)


@clinica.tool(name="vacunas")
def mcp_vacunas(mascota: Mascota, dueno: Dueno = "") -> dict:
    """Vacunas de una mascota: la próxima fecha de cada una y si está vencida."""
    return vacunas(mascota, dueno)


@clinica.tool(name="historial")
def mcp_historial(mascota: Mascota,
                  ultimas: Annotated[int, Field(description="Cuántas visitas.")] = 3,
                  dueno: Dueno = "") -> dict:
    """Las últimas visitas de una mascota: fecha, motivo, tratamiento y peso."""
    return historial(mascota, ultimas, dueno)


@clinica.tool(name="dosis")
def mcp_dosis(medicamento: Annotated[str, Field(description="Nombre del medicamento.")],
              mascota: Mascota, dueno: Dueno = "") -> dict:
    """Dosis de un medicamento según el peso actual, con avisos de alergia o especie."""
    return dosis(medicamento, mascota, dueno)


@clinica.tool(name="huecos_libres")
def mcp_huecos_libres(fecha: Fecha, veterinario: Annotated[str, Field(
        description="Nombre del veterinario; vacío para todos.")] = "") -> dict:
    """Horas libres de la agenda en un día, de un veterinario o de todos."""
    return huecos_libres(fecha, veterinario)


@clinica.tool(name="citas_de")
def mcp_citas_de(mascota: Mascota, dueno: Dueno = "") -> dict:
    """Citas de una mascota (pasadas y pendientes), con su id."""
    return citas_de(mascota, dueno)


@clinica.tool(name="reservar_cita")
def mcp_reservar_cita(mascota: Mascota,
                      veterinario: Annotated[str, Field(description="Nombre del veterinario.")],
                      fecha: Fecha,
                      hora: Annotated[str, Field(description="HH:MM, en punto o y media.")],
                      motivo: Annotated[str, Field(description="Motivo de la cita.")],
                      dueno: Dueno = "") -> dict:
    """Reserva una cita en una hora libre."""
    return reservar(mascota, veterinario, fecha, hora, motivo, dueno)


@clinica.tool(name="cancelar_cita")
def mcp_cancelar_cita(cita: Annotated[str, Field(
        description="Id de la cita, como CT-0026 (lo da citas_de).")]) -> dict:
    """Cancela una cita pendiente."""
    return cancelar_cita(cita)


@clinica.tool(name="avisar_dueno")
def mcp_avisar_dueno(mascota: Mascota,
                     texto: Annotated[str, Field(description="Texto del aviso, en español.")],
                     dueno: Dueno = "") -> dict:
    """Envía un aviso al dueño de una mascota, por correo o SMS según prefiera."""
    return avisar_dueno_de(mascota, texto, dueno)


facturacion = MCPServer("facturacion", instructions="Facturas de los clientes de la clínica.")


@facturacion.tool(name="cuentas")
def mcp_cuentas(dueno: Annotated[str, Field(description="Nombre del dueño.")]) -> dict:
    """Facturas sin pagar de un cliente, buscado por su nombre, y el total que debe."""
    return cuentas(dueno)

## 6. El anfitrión y el agente

In [ ]:
def sin_titulos(esquema):
    """El esquema sin las claves «title» que añade Pydantic y que el modelo no necesita."""
    if isinstance(esquema, dict):
        return {k: sin_titulos(v) for k, v in esquema.items() if k != "title"}
    return esquema


def compactar(texto):
    """El JSON de un resultado en una sola línea, sin la sangría con que lo envía el servidor."""
    try:
        return json.dumps(json.loads(texto), ensure_ascii=False)
    except ValueError:
        return texto


class Anfitrion:
    """Se conecta a varios servidores MCP, reúne sus herramientas y envía cada llamada
    al servidor que la ofrece. No sabe nada de la clínica. Con compacto, quita a los
    esquemas los títulos y escribe los resultados en una sola línea."""

    def __init__(self, servidores, compacto=False):
        self.servidores, self.compacto = servidores, compacto

    async def __aenter__(self):
        self.pila = AsyncExitStack()
        self.ruta, self.esquemas = {}, []
        for servidor in self.servidores:
            cliente = await self.pila.enter_async_context(Client(servidor))
            for h in (await cliente.list_tools()).tools:
                self.ruta[h.name] = cliente
                parametros = sin_titulos(h.input_schema) if self.compacto else h.input_schema
                self.esquemas.append({"type": "function", "function": {
                    "name": h.name, "description": h.description, "parameters": parametros}})
        return self

    async def __aexit__(self, *error):
        await self.pila.aclose()

    async def llamar(self, nombre, argumentos):
        """Llama a la herramienta en su servidor y devuelve el texto del resultado."""
        cliente = self.ruta.get(nombre)
        if cliente is None:
            return json.dumps({"error": f"No existe la herramienta {nombre}."}, ensure_ascii=False)
        r = await cliente.call_tool(nombre, argumentos)
        texto = "\n".join(b.text for b in r.content if b.type == "text")
        return compactar(texto) if self.compacto else texto


async def agente_mcp(tarea, anfitrion, max_pasos=8, ver=False):
    """El bucle del capítulo 1, con las herramientas de los servidores MCP."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, anfitrion.esquemas)
        llamadas = leer_llamadas(respuesta)[:1]
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "pensamiento": pensamiento,
                      "llamadas": []})
        if not llamadas:
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            texto = await anfitrion.llamar(llamada["name"], llamada["arguments"])
            traza[-1]["llamadas"].append({**llamada, "resultado": texto,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?", "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza

In [ ]:
async def probar():
    async with Anfitrion([clinica, farmacia, facturacion]) as a:
        print([e["function"]["name"] for e in a.esquemas])
        reiniciar()
        await agente_mcp("¿Cuánto cuesta la doxiciclina y la tenemos ahora?", a, ver=True)

await probar()

## 7. Dieciséis tareas y cuatro variantes

In [ ]:
def cifra(respuesta, valor):
    """¿Aparece la cifra, con coma o con punto y con uno o dos decimales?"""
    return dice(respuesta, *{f"{valor:.2f}".replace(".", "[,.]"),
                             f"{valor:.1f}".replace(".", "[,.]") + r"\b",
                             rf"\b{valor:g}\b".replace(".", "[,.]")})


FARMACIA_TAREAS = [
    ("doxiciclina", "¿Cuánto cuesta la doxiciclina y la tenemos ahora?",
     lambda r: cifra(r, 14.2) and dice(r, "no (hay|tenemos|queda)", "agotad", "sin existencias", r"\b0\b")),
    ("cuando_llega", "Se nos ha acabado la doxiciclina. ¿Cuándo llega?",
     lambda r: fecha_en(r, FARMACIA["doxiciclina"]["reposicion"])),
    ("meloxicam", "¿Cuántas unidades de meloxicam nos quedan?",
     lambda r: dice(r, rf"\b{FARMACIA['meloxicam']['existencias']}\b")),
    ("vacunas_de_toby", "¿Cuánto costaría ponerle a Toby todas las vacunas que tiene vencidas?",
     lambda r: importe(r, sum(TARIFAS["vacuna " + v["vacuna"]] for v in vacunas("Toby")["vacunas"]
                              if v["estado"] == "vencida"))),
]


BANCO = TAREAS + FARMACIA_TAREAS


def anotar(clave, comprobar, respuesta, traza, inicio):
    llamadas = [l for p in traza for l in p["llamadas"]]
    return {"tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
            "pasos": len(traza), "llamadas": [(l["name"], l["arguments"]) for l in llamadas],
            "tokens_resultados": sum(l["tokens_resultado"] for l in llamadas),
            "tokens_entrada": sum(p["tokens_entrada"] for p in traza),
            "segundos": round(time.time() - inicio, 1), "traza": traza}


def resolver_antes(tareas=BANCO):
    """El agente del capítulo 1 con las herramientas dentro del programa."""
    filas = []
    for clave, tarea, comprobar in tareas:
        reiniciar()
        inicio = time.time()
        respuesta, traza = agente(tarea, FUNCIONES, ESQUEMAS)
        filas.append(anotar(clave, comprobar, respuesta, traza, inicio))
    return filas


async def resolver_mcp(servidores, compacto=False, tareas=BANCO):
    """El mismo agente, con las herramientas de los servidores MCP."""
    filas = []
    async with Anfitrion(servidores, compacto) as anfitrion:
        for clave, tarea, comprobar in tareas:
            reiniciar()
            inicio = time.time()
            respuesta, traza = await agente_mcp(tarea, anfitrion)
            filas.append(anotar(clave, comprobar, respuesta, traza, inicio))
        esquemas = anfitrion.esquemas
    return filas, esquemas


def resumen(filas, esquemas):
    llamadas = sum(len(f["llamadas"]) for f in filas)
    return {"bien": sum(f["bien"] for f in filas), "pasos": sum(f["pasos"] for f in filas),
            "llamadas": llamadas,
            "tokens_por_resultado": round(sum(f["tokens_resultados"] for f in filas) / max(llamadas, 1)),
            "tokens_esquemas": len(tok(json.dumps(esquemas, ensure_ascii=False))["input_ids"]),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas)}

In [ ]:
todos = [clinica, farmacia, facturacion]
resultados = {}
for nombre, servidores, compacto in [
        ("Antes: herramientas en el programa", None, False),
        ("MCP: solo la clínica", [clinica], False),
        ("MCP: tres servidores", todos, False),
        ("MCP: tres servidores, anfitrión compacto", todos, True)]:
    if servidores is None:
        filas, esquemas = resolver_antes(), ESQUEMAS
    else:
        filas, esquemas = await resolver_mcp(servidores, compacto)
    resultados[nombre] = filas
    print(f"{nombre:42s}", resumen(filas, esquemas))

## Ejercicios

1. Explica con tus palabras qué hacen el anfitrión, el cliente y el servidor en la clínica. ¿Cuántos clientes tiene el agente con los tres servidores?

2. Ejecuta `conversacion_cruda()` y localiza en la respuesta a `initialize` las capacidades del servidor. ¿Qué crees que significa `listChanged`?

3. Compara el esquema de `existencias` escrito a mano en el antes con el que llega por `tools/list`. ¿Qué claves sobran? ¿Cuántos tokens ocupa cada uno?

4. Haz que una herramienta del servidor de la farmacia devuelva un texto ya compacto, una cadena con `json.dumps`, en lugar de un diccionario. ¿Cambia lo que llega al cliente? ¿Es mejor arreglarlo en el servidor o en el anfitrión?

5. Añade a la farmacia una herramienta `medicamentos_para(especie)` que liste los medicamentos indicados en una especie. ¿Qué tienes que tocar en el agente para que la use?

6. Lanza el agente solo con el servidor de la farmacia y pregúntale la dosis de meloxicam para Max. ¿Qué responde? ¿Inventa algo?

7. Mide cuánto tarda una llamada a `existencias` por stdio y otra a `cuentas` en el mismo programa, con `time.perf_counter`, repitiendo cien veces. ¿Importa la diferencia frente a lo que tarda el modelo en cada paso?

8. ¿Qué pasa si dos servidores publican una herramienta con el mismo nombre? Pruébalo añadiendo `precio` también al servidor de la facturación. ¿A cuál llega la llamada en el `Anfitrion` del capítulo?

9. Sigue la documentación de un asistente de escritorio que admita servidores MCP y configúralo para que lance el servidor de la farmacia, y hazle las cuatro preguntas de farmacia. Compara sus respuestas con las del agente del libro.

10. **Proyecto.** Convierte en servidor MCP una herramienta que uses en tu trabajo o en tus estudios, con dos o tres funciones, y conéctala al agente del capítulo junto a los de la clínica. Escribe tres tareas que la necesiten y mide el agente con y sin ella.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Especificación de MCP](https://modelcontextprotocol.io/specification) y [SDK de Python](https://github.com/modelcontextprotocol/python-sdk). [Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol) (Anthropic, 2024). Lección 13.06 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.